In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show

# 1. 載入資料

## 新增表格title

In [3]:
import pandas as pd
from interpret.glassbox import ExplainableBoostingClassifier

column_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
                'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

## 讀取

In [4]:
df = pd.read_csv('../data/heart.csv', names=column_names, na_values='?')

## 處理缺失值

In [5]:
df = df.dropna()

## 轉換標籤

In [6]:
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

# 2. 分割特徵與標籤

In [7]:
y = df["target"]
X = df.drop(columns=["target"])

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


# 3. 訓練/測試分割

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 4. 訓練 EBM

In [9]:
ebm = ExplainableBoostingClassifier(random_state=42)
ebm.fit(X_train, y_train)

,feature_names,None
,feature_types,None
,max_bins,1024
,max_interaction_bins,64
,interactions,'3x'
,exclude,None
,validation_size,0.15
,outer_bags,14
,inner_bags,0
,learning_rate,0.015
,greedy_ratio,10.0


# 5. 計算 AUC

In [10]:
y_prob = ebm.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
print(f"EBM Test AUC:{auc:.3f}")

EBM Test AUC:0.940


# 6. 全域解釋 — 看到所有特徵的 shape function

In [11]:
ebm_global = ebm.explain_global()
show(ebm_global)

<!-- http://127.0.0.1:7001/5854605536/ -->

# 7. 局部解釋 — 看到前 5 位病患的預測分解

In [12]:
ebm_local = ebm.explain_local(X_test[:5], y_test[:5])
show(ebm_local)

<!-- http://127.0.0.1:7001/5884906704/ -->